# U06 | 词向量与 RNN 基础

**本单元目标**：从处理表格数据跨越到处理文本序列，理解：

1. 词向量（Word Embedding）—— 词语如何变成数字
2. RNN 基础 —— 如何处理变长序列
3. BPTT —— RNN 的梯度是怎么传的

---

## 1. 从分类到序列：任务变了

前面的任务都是：**一个样本 -> 一个标签**

```
图片(784,) -> 数字0~9
表格数据 -> 类别
```

现在要做翻译：

```
我爱学习  ->  I love learning
我喜欢猫  ->  I like cats
学习真好  ->  Learning is great
```

输入是**变长序列**，输出也是**变长序列**。

这需要全新的建模方式。

## 2. 词语怎么变成数字？—— One-Hot 编码

最直观的做法：**给每个词分配一个索引，用 one-hot 向量表示**

假设词表大小 VOCAB = 5：

| 词 | 索引 | One-Hot |
|---|------|---------|
| 我 | 0 | [1, 0, 0, 0, 0] |
| 爱 | 1 | [0, 1, 0, 0, 0] |
| 学 | 2 | [0, 0, 1, 0, 0] |
| 习 | 3 | [0, 0, 0, 1, 0] |
| 好 | 4 | [0, 0, 0, 0, 1] |

**问题**：词表 1 万个词 -> 每个词是 1 万维向量，**99.99% 都是 0**。极度稀疏，且词之间毫无语义关联。

```python
import torch

vocab_size = 10000
word_index = torch.tensor([0, 1, 2, 3])   # 句子：我爱学习

one_hot = torch.zeros(word_index.shape[0], vocab_size)
for i, idx in enumerate(word_index):
    one_hot[i, idx] = 1

print(one_hot.shape)   # (4, 10000)
print(one_hot[0])      # 只有第0位是1，其他全是0

## 3. 词向量（Word Embedding）：从稀疏到稠密

**核心思想**：不再用 1 万维的 one-hot，而是用一个小得多的稠密向量，比如 64 维或 128 维。

```
我  ->  [0.23, -0.45, 0.87, ..., 0.12]   # 64维向量
爱  ->  [0.56, 0.12, -0.33, ..., 0.78]
学  ->  [-0.11, 0.89, 0.45, ..., -0.23]
```

这组向量是**可学习的参数**，模型会在训练中自动调整，让语义相近的词，向量也相近。

**PyTorch 实现**：`nn.Embedding`

```python
import torch
import torch.nn as nn

vocab_size = 10000   # 词表大小
embed_dim = 64       # 每个词的向量维度

embedding = nn.Embedding(vocab_size, embed_dim)
print(embedding.weight.shape)   # (10000, 64)

word_index = torch.tensor([0, 1, 2, 3])   # 句子：我爱学习（4个词）
vecs = embedding(word_index)               # (4, 64)

print(vecs.shape)   # 4个词，每个64维向量
print(vecs[0])      # 词'我'的64维向量

In [2]:
import torch
import torch.nn as nn

vocab_size = 10000   # 词表大小
embed_dim = 64       # 每个词的向量维度

embedding = nn.Embedding(vocab_size, embed_dim)
print(embedding.weight.shape)   # (10000, 64)

word_index = torch.tensor([0, 1, 2, 3])   # 句子：我爱学习（4个词）
print(word_index)
vecs = embedding(word_index)               # (4, 64)

print(vecs.shape)   # 4个词，每个64维向量
print(vecs[0])      # 词'我'的64维向量

torch.Size([10000, 64])
tensor([0, 1, 2, 3])
torch.Size([4, 64])
tensor([ 1.6712, -0.0654,  0.4268, -2.5001, -0.8898, -0.2454, -2.3185, -0.1251,
        -0.7651, -1.1816,  0.2969, -0.4720,  0.6689,  0.6982,  1.2398,  1.7097,
         1.3174,  0.3273,  0.8822,  0.1157,  0.7376,  0.6786, -0.2690, -1.0110,
         1.1638, -0.1430, -0.2104,  0.2679,  0.2490,  0.0696, -1.3989, -0.8397,
        -1.5096, -0.2354, -0.9519,  1.0261,  1.3951, -0.3408, -0.3938, -0.7551,
        -0.5616, -1.2563,  0.1281, -0.4852, -0.9303, -0.4748,  0.0925,  0.3490,
         0.3918, -0.4032,  0.2546, -0.4085,  0.9601, -0.8745,  0.5157, -0.0496,
        -1.1414,  0.8509,  1.4437,  1.6035, -1.3290,  0.3932,  2.7568, -1.1162],
       grad_fn=<SelectBackward0>)

## 4. nn.Embedding 的工作原理：一张可学习的查表

上一节我们用 `nn.Embedding(10000, 64)` 创建了一个 Embedding 层，但它内部到底存了什么？

### 4.1 Embedding 的本质：一张词向量表

`nn.Embedding(vocab_size, embed_dim)` 做的**唯一一件事**：创建一张 `(vocab_size, embed_dim)` 的可学习矩阵。

用一个小例子看清楚：

```python
embedding = nn.Embedding(5, 3)   # 词表5个词，每个词3维向量
print(embedding.weight)
# tensor([[ 0.12, -0.34,  0.56],   <- 词0 的向量
#         [ 0.78,  0.23, -0.45],   <- 词1 的向量
#         [-0.11,  0.89,  0.12],   <- 词2 的向量
#         [ 0.45, -0.67,  0.33],   <- 词3 的向量
#         [-0.23,  0.56,  0.78]])  <- 词4 的向量
```

这张表就是 **Embedding 的全部内容**。每一行就是一个词的向量。

### 4.2 调用 Embedding：按索引取行

用 `embedding(idx)` 就是根据 idx 里的数字，从表里取对应的行：

```python
idx = torch.tensor([1, 2, 0])
vec = embedding(idx)
# vec 是：
# tensor([[ 0.78,  0.23, -0.45],   <- 词1 的向量
#         [-0.11,  0.89,  0.12],   <- 词2 的向量
#         [ 0.12, -0.34,  0.56]])  <- 词0 的向量
print(vec.shape)   # (3, 3) - 取了3行，每行3维
```

**等价于 fancy indexing**：
```python
embedding(idx)  ==  embedding.weight[idx]
```

### 4.3 输入形状：传几个索引，吐几个向量

Embedding 不在乎输入的维度语义，规则很简单：**输入 shape `(...)` -> 输出 shape `(..., embed_dim)`**，最后多一维就是词向量。

```python
# 1D：3个查询 -> 3个向量
embedding(torch.tensor([1, 2, 0])).shape          # (3, 3)

# 2D：(batch=2, seq_len=4) 的句子 -> (2, 4, 3)
embedding(torch.tensor([[1, 2, 0, 3],
                        [4, 1, 2, 0]])).shape    # (2, 4, 3)
```

在 NLP 训练中，约定输入 shape 是 `(batch, seq_len)`，每个数字是词索引。

### 4.4 类比 nn.Linear，理解差异

| 层 | 输入 | 输出 | 计算方式 |
|---|------|------|----------|
| `nn.Linear(in, out)` | 实数向量 `(batch, in)` | `(batch, out)` | 矩阵乘 + 偏置 |
| `nn.Embedding(vocab, dim)` | 整数索引 `(batch, seq_len)` | `(batch, seq_len, dim)` | 查表（取行）|

最大区别：**Embedding 输入的是整数索引，Linear 输入的是实数向量**。

### 4.5 为什么这张表是可学习的？

`embedding.weight` 是 `nn.Parameter`，会被 `optimizer.step()` 更新：

- 训练开始时：表里都是随机数，词之间没有任何关系
- 训练过程中：根据 loss 反向传播，逐渐调整每个词的向量
- 训练结束后：语义相近的词，向量也接近（比如猫和狗的向量距离会很近）

这就是为什么我们叫它词嵌入——每个词被嵌入到了一个有语义结构的低维空间里。

## 5. RNN 是什么？

RNN（循环神经网络）专门处理**序列数据**：

核心思想：**每读一个词，就更新一次记忆（隐藏状态）**

```
词1 -> RNN -> h1 -> RNN -> h2 -> RNN -> h3 -> ...
  ^                              |
  ---------- 上一时刻的隐藏状态 <-+
```

**三个核心变量**：

| 变量 | 含义 | 形状 |
|---|------|------|
| x_t | t 时刻的输入（词向量）| (batch, embed_dim) |
| h_t | t 时刻的隐藏状态 | (batch, hidden_dim) |
| h_{t-1} | 上一时刻的隐藏状态 | (batch, hidden_dim) |

**RNN 计算公式（数学形式）**：

$$h_t = \tanh(W_{xh} \cdot x_t + W_{hh} \cdot h_{t-1} + b)$$

**拆开看每一项**：

| 项 | 含义 |
|---|------|
| $W_{xh} \cdot x_t$ | 当前输入词向量经过线性变换 |
| $W_{hh} \cdot h_{t-1}$ | 上一时刻的记忆经过线性变换 |
| $b$ | 偏置项 |
| $\tanh(\cdot)$ | 激活函数，把结果压到 (-1, 1) |

**对应的 PyTorch 代码**：
```python
h_t = torch.tanh(x_t @ W_xh.T + h_prev @ W_hh.T + b)
```

**直观含义**：综合当前词的信息和之前积累的记忆，产生新的记忆。

---

**为什么要加 tanh？** 假设没有 tanh：

$$h_t = W_{xh} x_t + W_{hh} h_{t-1} + b$$

展开几步：

$$h_1 = W_{xh} x_1 + W_{hh} h_0$$

$$h_2 = W_{xh} x_2 + W_{hh} h_1 = W_{xh} x_2 + W_{hh}(W_{xh} x_1 + W_{hh} h_0)$$

$$h_2 = W_{xh} x_2 + W_{hh}W_{xh} x_1 + W_{hh}^2 h_0$$

本质上还是个线性变换——不管多少层，都等价于一个矩阵乘法。**tanh 提供两个作用**：① 引入非线性 ② 把结果压到 (-1, 1)，防止反复传递时数值爆炸。

PyTorch 提供了 `nn.RNN`，我们先学会使用，再理解内部原理。

## 6. PyTorch 的 nn.RNN

```python
import torch
import torch.nn as nn

rnn = nn.RNN(
    input_size=64,     # 每个词的向量维度
    hidden_size=128,   # 隐藏状态的维度
    num_layers=1,     # RNN 层数
    batch_first=True  # 输入格式：(batch, seq_len, input_size)
)
print([n.shape for n in rnn.named_parameters()])
# [('weight_ih_l0', (128, 64)),   # W_xh
#  ('weight_hh_l0', (128, 128)),  # W_hh
#  ('bias_ih_l0', (128,)),
#  ('bias_hh_l0', (128,))]

## 7. RNN 的输入输出

```python
batch_size = 4
seq_len = 6           # 每个句子6个词
input_size = 64       # 词向量维度

x = torch.randn(batch_size, seq_len, input_size)

h0 = torch.zeros(1, batch_size, 128)

output, hn = rnn(x, h0)

print('output:', output.shape)   # (4, 6, 128) 每个时刻的隐藏状态
print('hn:', hn.shape)          # (1, 4, 128) 最后一个时刻的隐藏状态

## 8. 输出解读：output vs hn

```
output: 包含所有6个时刻的隐藏状态
  output[:, 0, :]  -> 读词1后的h1
  output[:, 1, :]  -> 读词2后的h2
  ...
  output[:, 5, :]  -> 读词6后的h6

hn: 只保留最后一个时刻的隐藏状态
  hn[0, :, :]      -> h6（整个句子的最终记忆）
```

**常见用法**：
- 序列分类：取最后一个隐藏状态 hn，接分类层
- 序列生成：取所有 output，一步一步预测下一个词
- 序列标注：取每个时刻的 output，分别做预测

## 9. 简单序列分类：Embedding + RNN + 分类

把词嵌入和 RNN 拼在一起，处理一个句子的情感分类（正面/负面）：

```python
class SentenceClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 2)   # 二分类

    def forward(self, x):
        # x: (batch, seq_len)  词索引
        vecs = self.embedding(x)             # (batch, seq_len, embed_dim)
        _, hn = self.rnn(vecs)               # hn: (1, batch, hidden_dim)
        out = self.fc(hn.squeeze(0))         # (batch, 2)
        return out

model = SentenceClassifier(vocab_size=10000, embed_dim=64, hidden_dim=128)

sentence = torch.randint(0, 10000, (8, 5))   # 8个句子，每个5个词
logits = model(sentence)
print(logits.shape)   # (8, 2)

## 10. BPTT：RNN 的梯度是怎么传的

RNN 的反向传播叫 **BPTT（Backpropagation Through Time）**——时间维度上的反向传播。

原理：把 RNN 按时间展开，每个时刻都有一份参数 W_xh 和 W_hh。

```
时刻1: h1 = tanh(x1*W + h0*W)
时刻2: h2 = tanh(x2*W + h1*W)
时刻3: h3 = tanh(x3*W + h2*W)
```

反向传播时：

```
dL/dh3 -> dL/dh2 -> dL/dh1 -> dL/dW
```

**梯度消失问题**：如果序列很长，梯度要穿越很多层 tanh，每一层都乘一个小于1的数 -> 越来越小 -> 前面的词几乎收不到梯度。

这就是 **LSTM/GRU** 出现的原因——它们用门控机制让梯度更容易传递。

**好消息**：PyTorch 的 `rnn.backward()` 自动处理这些，训练时写法和 MLP 一样：

```python
optimizer.zero_grad()
logits = model(sentence)
loss = nn.CrossEntropyLoss()(logits, labels)
loss.backward()      # BPTT 自动完成
optimizer.step()
```

## 11. 变长序列：Padding 与 Packing

真实数据中，每个句子长度不同：

```
句子1: 我 爱 学 习                    -> 4个词
句子2: 今 天 天 气 很 好               -> 6个词
句子3: 谢 谢                           -> 2个词
```

两种处理方式：

**方式1：Padding（填充）**：
把短句补0，长度统一到 max_len。简单但低效。

```python
padded = torch.nn.utils.rnn.pad_sequence([seq1, seq2, seq3], batch_first=True, padding_value=0)
# 得到 (3, max_len, embed_dim)

**方式2：Packing（打包）**：只保留有效长度，不浪费计算。

```python
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

padded = torch.nn.utils.rnn.pad_sequence([seq1, seq2, seq3], batch_first=True, padding_value=0)

lengths = torch.tensor([4, 6, 2])   # 三个句子的真实长度
packed = pack_padded_sequence(padded, lengths.cpu(), batch_first=True, enforce_sorted=False)

output_packed, hn = rnn(packed)

output, _ = pad_packed_sequence(output_packed, batch_first=True)
print(output.shape)   # (3, 6, hidden_dim)

## 12. 本单元小结

| 概念 | 作用 |
|---|------|
| One-Hot | 最直观的词表示，但稀疏、无语义 |
| nn.Embedding | 可学习的稠密词向量，语义相似的词向量也相近 |
| nn.RNN | 处理序列，每步融合当前词和历史记忆 |
| BPTT | RNN 的反向传播，梯度沿时间展开传递 |
| 梯度消失 | 长序列时前面词的梯度衰减 -> LSTM/GRU 的动机 |
| Packing | 处理变长序列，省计算 |

下一单元，我们把 RNN 换成 **GRU**，加入 **Attention 机制**，然后实现完整的翻译模型。